# PHASE GPU-C: Google Colab Remote GPU Inference Server

This notebook launches the complete computer-vision serving layer (**FastAPI + OpenCLIP ViT-B/32 + Google OWL-ViT + Phase 6C Two-Stage Region Pipeline**) on a high-performance **NVIDIA CUDA GPU** in Google Colab.

### Architecture:
```
+-------------------------+         HTTPS Multi-part         +-------------------------------+
|   LOCAL LAPTOP          | -------------------------------> |   GOOGLE COLAB (NVIDIA GPU)   |
|   - Antigravity IDE     |                                  |   - FastAPI (Port 8000)       |
|   - Next.js 14 Frontend | <------------------------------- |   - OpenCLIP (CUDA)           |
|   - Local Image Upload  |          JSON Predictions        |   - OWL-ViT (CUDA)            |
|   - Canvas Bounding Box |                                  |   - Phase 6C Pipeline (CUDA)  |
+-------------------------+                                  +-------------------------------+
```

> **Prerequisite:** Ensure Colab is using a GPU runtime: `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`.

## CELL 1: GPU Diagnostics & CUDA Verification
Detects host hardware, driver readiness, PyTorch CUDA build, and available VRAM.

In [ ]:
import sys
import os
import platform
import torch

print("=" * 70)
print("ENVIRONMENT & HARDWARE DIAGNOSTICS")
print("=" * 70)
print(f"Python Version:     {sys.version.split()[0]} ({platform.system()} {platform.release()})")
print(f"PyTorch Version:    {torch.__version__}")
print(f"CUDA Available:     {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print("\n[ERROR] CRITICAL: No NVIDIA GPU / CUDA runtime detected!")
    print("Please navigate to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run.")
    raise RuntimeError("GPU runtime required. Current runtime is CPU-only.")

gpu_count = torch.cuda.device_count()
current_device = torch.cuda.current_device()
gpu_name = torch.cuda.get_device_name(current_device)
vram_total_gb = torch.cuda.get_device_properties(current_device).total_memory / (1024 ** 3)
cuda_version = torch.version.cuda

print(f"CUDA Runtime:       v{cuda_version}")
print(f"GPU Device:         {gpu_name} (Index: {current_device}, Count: {gpu_count})")
print(f"Total VRAM:         {vram_total_gb:.2f} GB")
print("=" * 70)
print("[SUCCESS] NVIDIA GPU verified and ready for hardware-accelerated inference.")

## CELL 2: Repository Setup & Workspace Directory
Clones or synchronizes the project repository into the Colab environment.

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------------
# CONFIGURATION: Set Repository URL
# ---------------------------------------------------------------------------
# Replace with your Git repository URL if cloning remotely.
# If uploading manually or via Google Drive, set REPO_DIR to the project root.
GIT_REPO_URL = ""  # e.g., "https://github.com/your-org/ml-project.git"
REPO_NAME = "ml-project"
WORKSPACE_DIR = Path("/content") / REPO_NAME if "/content" in sys.path or Path("/content").exists() else Path.cwd()

if GIT_REPO_URL and not WORKSPACE_DIR.exists():
    print(f"Cloning repository from {GIT_REPO_URL}...")
    !git clone {GIT_REPO_URL} {WORKSPACE_DIR}
else:
    print(f"Using active workspace directory: {WORKSPACE_DIR}")

if str(WORKSPACE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_DIR))
    print(f"Added {WORKSPACE_DIR} to Python sys.path")

os.chdir(WORKSPACE_DIR)
print(f"Current working directory: {Path.cwd()}")

## CELL 3: Dependency Installation
Installs required packages without overwriting the pre-installed CUDA PyTorch.

In [ ]:
print("Installing project dependencies...")
!pip install -q open-clip-torch transformers fastapi "uvicorn[standard]" python-multipart httpx pydantic pillow

# Verify critical packages
import open_clip
import transformers
import fastapi
import uvicorn

print(f"open_clip version:     {open_clip.__version__}")
print(f"transformers version:  {transformers.__version__}")
print(f"fastapi version:       {fastapi.__version__}")
print(f"uvicorn version:       {uvicorn.__version__}")
print("[SUCCESS] All dependencies satisfied.")

## CELL 4: Force GPU Environment Configuration
Sets `ML_DEVICE=cuda` and verifies the centralized device resolver.

In [ ]:
os.environ["ML_DEVICE"] = "cuda"
os.environ["OWL_VIT_DEVICE"] = "cuda"
os.environ["OPENCLIP_DEVICE"] = "cuda"

from src.device import get_device_info, resolve_device

resolved = resolve_device()
dev_info = get_device_info()

print("=" * 70)
print("DEVICE RESOLUTION VERIFICATION")
print("=" * 70)
print(f"Target Device Object:    {resolved}")
print(f"Selected Device String:  {dev_info['selected_device']}")
print(f"CUDA Available:          {dev_info['cuda_available']}")
print(f"GPU Model Name:          {dev_info['cuda_device_name']}")
print("=" * 70)

assert resolved.type == "cuda", f"Device resolution failed to select CUDA (got {resolved})"

## CELL 5: Model & Tensor Device Verification
Loads OpenCLIP, OWL-ViT, and Phase 6C pipeline on CUDA to verify GPU tensor placement and forward pass.

In [ ]:
import time
from PIL import Image
import numpy as np
from src.phase6.open_vocab_classifier import OpenVocabClassifier
from src.phase6.owlvit.detector import OWLViTDetector
from src.phase6.region_pipeline import RegionRecognitionPipeline

print("1. Initializing OpenCLIP ViT-B/32 on CUDA...")
t0 = time.perf_counter()
clip_clf = OpenVocabClassifier(device="cuda")
print(f"   OpenCLIP initialized on {clip_clf.device} in {(time.perf_counter()-t0)*1000:.1f}ms")

print("2. Initializing Google OWL-ViT on CUDA...")
t0 = time.perf_counter()
owl_det = OWLViTDetector(device="cuda")
print(f"   OWL-ViT initialized on {owl_det.device} in {(time.perf_counter()-t0)*1000:.1f}ms")

print("3. Verifying Two-Stage Region Pipeline on CUDA...")
pipeline = RegionRecognitionPipeline(detector=owl_det, classifier=clip_clf)

# Create synthetic test image
dummy_image = Image.fromarray(np.uint8(np.random.randint(0, 255, (480, 640, 3))))
t0 = time.perf_counter()
test_out = pipeline.process_image(
    image_input=dummy_image,
    text_queries=["object", "pattern", "texture"],
    max_regions=5
)
elapsed = (time.perf_counter() - t0) * 1000
print(f"   Phase 6C Execution completed in {elapsed:.1f}ms on {test_out['models']['device']}.")
print("[SUCCESS] Models successfully resident on CUDA.")

## CELL 6: Start FastAPI Serving Layer in Background
Starts Uvicorn serving `src.api.main:app` as a background daemon process on port 8000.

In [ ]:
import subprocess
import time
import httpx

# Kill any existing uvicorn process
!pkill -f uvicorn || true
time.sleep(1)

log_file = open("fastapi_server.log", "w")
print("Starting FastAPI Uvicorn Server in background...")
server_process = subprocess.Popen(
    ["uvicorn", "src.api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ, "ML_DEVICE": "cuda"}
)

# Poll health endpoint until server is ready
server_ready = False
for attempt in range(30):
    time.sleep(1)
    try:
        r = httpx.get("http://127.0.0.1:8000/api/v1/health", timeout=2.0)
        if r.status_code == 200:
            server_ready = True
            print(f"[SUCCESS] FastAPI Server is LIVE and healthy (PID: {server_process.pid})")
            break
    except Exception:
        pass

if not server_ready:
    print("[ERROR] Server failed to start within 30 seconds. Logs:")
    with open("fastapi_server.log", "r") as f:
        print(f.read())
    raise RuntimeError("FastAPI startup timeout")

## CELL 7: Health & Model Telemetry Verification
Validates `/api/v1/health`, `/api/v1/ready`, and `/api/v1/models` against the live server.

In [ ]:
import json

client = httpx.Client(base_url="http://127.0.0.1:8000")

health_resp = client.get("/api/v1/health").json()
ready_resp = client.get("/api/v1/ready").json()
models_resp = client.get("/api/v1/models").json()

print("=" * 70)
print("LIVE FASTAPI SERVER TELEMETRY")
print("=" * 70)
print(f"Service Status:          {health_resp.get('status')}")
print(f"Models Ready:            {ready_resp.get('models_ready')}")
print(f"OpenCLIP Device:         {models_resp['open_vocabulary']['device']}")
print(f"OWL-ViT Device:          {models_resp['open_vocabulary_detection']['device']}")
print(f"Hardware Device Info:    {json.dumps(models_resp.get('device_info', {}), indent=2)}")
print("=" * 70)

## CELL 8: Secure Tunnel Configuration (`cloudflared`)
Exposes the FastAPI port (8000) over an encrypted HTTPS tunnel for the local Next.js client.

In [ ]:
# Download Cloudflare tunnel binary if not present
!if [ ! -f /usr/local/bin/cloudflared ]; then curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared; fi

# Terminate any prior tunnel
!pkill -f cloudflared || true
time.sleep(1)

tunnel_log = open("tunnel.log", "w")
print("Launching Cloudflare Tunnel on port 8000...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT
)

# Extract public HTTPS URL from tunnel logs
import re
public_url = None
for attempt in range(25):
    time.sleep(1)
    with open("tunnel.log", "r") as f:
        content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print("\n" + "=" * 70)
    print("PUBLIC DEVELOPMENT API TUNNEL IS LIVE")
    print("=" * 70)
    print(f"\nCOLAB_API_URL = {public_url}\n")
    print("NEXT STEPS FOR LOCAL LAPTOP:")
    print(f"1. Open 'frontend/.env.local' on your laptop.")
    print(f"2. Set: NEXT_PUBLIC_API_BASE_URL={public_url}")
    print(f"3. Restart Next.js dev server: npm run dev")
    print("=" * 70)
else:
    print("[WARNING] Could not automatically parse tunnel URL. Check tunnel.log:")
    with open("tunnel.log", "r") as f:
        print(f.read())

## CELL 9: Direct ML Endpoint Testing & Verification
Executes end-to-end multi-part HTTP requests against the live serving layer.

In [ ]:
import io

# Generate synthetic test image with colorful geometric shapes
canvas = Image.new("RGB", (640, 480), color=(240, 240, 240))
from PIL import ImageDraw
draw = ImageDraw.Draw(canvas)
draw.rectangle([50, 50, 250, 300], fill=(220, 50, 50), outline=(0, 0, 0))
draw.ellipse([350, 100, 550, 350], fill=(50, 120, 220), outline=(0, 0, 0))

img_byte_arr = io.BytesIO()
canvas.save(img_byte_arr, format="JPEG")
img_bytes = img_byte_arr.getvalue()

test_url = public_url if public_url else "http://127.0.0.1:8000"
print(f"Testing endpoints on {test_url}...")

# 1. Test /api/v1/open-vocabulary
print("\n--- 1. Testing POST /api/v1/open-vocabulary ---")
files = {"image": ("test.jpg", img_bytes, "image/jpeg")}
data = {"text_queries": "red box, blue circle, green tree, golden retriever"}
r1 = httpx.post(f"{test_url}/api/v1/open-vocabulary", files=files, data=data, timeout=10.0)
res1 = r1.json()
print(f"Status: {r1.status_code} | Inference: {res1['open_vocabulary']['inference_ms']}ms")
print(f"Top Match: {res1['open_vocabulary']['top_match']}")

# 2. Test /api/v1/open-vocabulary/detect
print("\n--- 2. Testing POST /api/v1/open-vocabulary/detect ---")
files = {"image": ("test.jpg", img_bytes, "image/jpeg")}
data = {"text_queries": "red box, blue circle", "score_threshold": "0.10"}
r2 = httpx.post(f"{test_url}/api/v1/open-vocabulary/detect", files=files, data=data, timeout=15.0)
res2 = r2.json()
print(f"Status: {r2.status_code} | Total Detections: {res2['summary']['total_detections']}")
print(f"Timing: {res2['timing']}")

# 3. Test /api/v1/open-vocabulary/region-recognition
print("\n--- 3. Testing POST /api/v1/open-vocabulary/region-recognition ---")
files = {"image": ("test.jpg", img_bytes, "image/jpeg")}
data = {"text_queries": "red box, blue circle, yellow star", "max_regions": "5"}
r3 = httpx.post(f"{test_url}/api/v1/open-vocabulary/region-recognition", files=files, data=data, timeout=20.0)
res3 = r3.json()
print(f"Status: {r3.status_code} | Analyzed Regions: {res3['summary']['analyzed_regions_count']}")
print(f"Pipeline Timing: {res3['timing']}")
print("\n[SUCCESS] All ML API endpoints functional.")

## CELL 10: GPU Memory Telemetry & Empirical Benchmarking
Inspects actual allocated GPU VRAM and performs a controlled latency benchmark across query and region counts.

In [ ]:
print("=" * 70)
print("GPU MEMORY PROFILE & EMPIRICAL BENCHMARK")
print("=" * 70)

alloc_mb = torch.cuda.memory_allocated() / (1024 ** 2)
res_mb = torch.cuda.memory_reserved() / (1024 ** 2)
max_alloc_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

print(f"Current Allocated VRAM:  {alloc_mb:.1f} MB")
print(f"Current Reserved VRAM:   {res_mb:.1f} MB")
print(f"Peak Allocated VRAM:     {max_alloc_mb:.1f} MB")
print("=" * 70)

print("\nExecuting Measured Latency Benchmark on NVIDIA GPU...")
query_matrix = [1, 5, 10, 20]
benchmark_results = []

candidate_labels = [
    "dog", "cat", "car", "tree", "person", "chair", "bottle", "laptop",
    "cellphone", "table", "cup", "book", "door", "window", "shoe",
    "clock", "bird", "bicycle", "flower", "box"
]

for q_count in query_matrix:
    queries = candidate_labels[:q_count]
    
    # OpenCLIP Benchmark
    t_start = time.perf_counter()
    clip_res = clip_clf.classify_zero_shot(canvas, queries)
    clip_lat = (time.perf_counter() - t_start) * 1000
    
    # OWL-ViT Benchmark
    t_start = time.perf_counter()
    owl_res = owl_det.detect_objects(canvas, queries)
    owl_lat = (time.perf_counter() - t_start) * 1000
    
    # Phase 6C Benchmark
    t_start = time.perf_counter()
    pipe_res = pipeline.process_image(canvas, queries, max_regions=q_count)
    pipe_lat = (time.perf_counter() - t_start) * 1000
    
    benchmark_results.append({
        "queries": q_count,
        "openclip_ms": round(clip_lat, 2),
        "owlvit_ms": round(owl_lat, 2),
        "phase6c_ms": round(pipe_lat, 2),
    })

print("\n| Queries | OpenCLIP ViT-B/32 (ms) | OWL-ViT Base (ms) | Phase 6C Two-Stage (ms) |")
print("|---|---|---|---|")
for row in benchmark_results:
    print(f"| {row['queries']:<7} | {row['openclip_ms']:<22} | {row['owlvit_ms']:<17} | {row['phase6c_ms']:<23} |")
print("=" * 70)

## CELL 11: Shutdown & Resource Cleanup
Terminates background services, unloads models, and reclaims GPU memory.

In [ ]:
print("Shutting down serving background processes...")
!pkill -f cloudflared || true
!pkill -f uvicorn || true

import gc
from src.phase6.owlvit.lifecycle import OWLViTLifecycleManager

OWLViTLifecycleManager.unload()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

vram_after = torch.cuda.memory_allocated() / (1024 ** 2)
print(f"Reclaimed GPU VRAM. Remaining allocated: {vram_after:.1f} MB")
print("[SUCCESS] Background processes stopped and GPU VRAM cleanly released.")